In [ ]:
import os 
import xarray as xr
from joblib import Parallel, delayed
import multiprocessing
import glob
import pandas as pd

In [ ]:
z_s = xr.open_dataset("/work/uc1275/u301827/02_MSE/surface_geopotential_zs.nc")

In [ ]:
PARIS_LON = 13.4 #paris 2.35
PARIS_LAT = 52.52#paris 48.86
z_surface = z_s.z.sel(longitude=PARIS_LON, method = "nearest")\
                .sel(latitude=PARIS_LAT, method = 'nearest').values[0]

In [ ]:
# === Edit these paths if different ===
final_path = "/work/uc1275/u301827/02_MSE/berlin/raw/"

# Variable mapping (as you provided)
era5_vars = {
    130: "t",   # Temperature, pl @ 500 hPa
    129: "z",   # Geopotential, pl @ 500 hPa
    133: "q",   # Specific humidity, ml @ level 137
    167: "tasmax",  # 2m temperature, surface (no levels)
    166: "2t",  # 2m temperature (surface field)
    168: "2d",  # 2m temperature (surface field)
    39: "swvl1",
    #232: "ie",
    #59: "cape",
    159: "blh",
    #182: "e",
    #507: "pev",
    #146: "sshf",
    #147: "slhf",
    #49: "10fg",
    134: "sp"
    
}

def _find_sel_file(var_name, date_str, base_dir=final_path):
    """Return the path to sel_{var}_{YYYY-MM-DD}_level.nc or raise FileNotFoundError."""
    pattern = os.path.join(base_dir, var_name, f"{var_name}_{date_str}_paris.nc")
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(f"No file found for variable '{var_name}' and date '{date_str}'. looked for: {pattern}")
    if len(matches) > 1:
        # pick first but warn
        print(f"Warning: multiple matches for {pattern}. Using {matches[0]}")
    return matches[0]

def open_era5_day(date, era5_vars=era5_vars, final_path=final_path):
    """
    Open all preprocessed ERA5 variable files for one day plus the surface geopotential file.
    Parameters
    ----------
    date : str or pd.Timestamp
      e.g. "2021-06-21" or pd.Timestamp("2021-06-21")
    Returns
    -------
    dict with keys:
      - 'datasets': dict mapping var_name -> xarray.Dataset (opened)
      - 'zs': xarray.DataArray or Dataset (surface geopotential, time removed if present)
      - 'date_str': the 'YYYY-MM-DD' string used
    """
    if isinstance(date, pd.Timestamp):
        date_str = date.strftime("%Y-%m")
    else:
        # accept "YYYY-MM-DD" or similar
        date_str = pd.to_datetime(date).strftime("%Y-%m")

    datasets = {}
    for varnum, varname in era5_vars.items():
        try:
            path = _find_sel_file(varname, date_str, base_dir=final_path)
        except FileNotFoundError as e:
            # re-raise with clearer message
            raise FileNotFoundError(f"Missing preprocessed file for var '{varname}' on {date_str}: {e}")
        ds = xr.open_dataset(path)

        # Round time to daily resolution (drop hours/minutes/seconds)
        ds["time"] = pd.to_datetime(ds["time"].values).normalize()
        
        # Or, if you want just YYYY-MM-DD without time at all
        ds["time"] = pd.to_datetime(ds["time"].values).floor("D")
        datasets[varname] = ds

    return {"datasets": datasets, "zs": z_surface, "date_str": date_str}

# -----------------------
# Example usage:
# res = open_era5_day("2021-06-21")
# ds_2t = res['datasets']['2t']
# ds_q  = res['datasets']['q']
# ds_t500 = res['datasets']['t']   # 500 hPa temperature (if your 't' file contains it)
# zs = res['zs']
# -----------------------

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units
import xarray as xr

def compute_surface_mse(res):
    """
    Compute surface moist static energy (MSE) from 2m temperature, specific humidity, and surface geopotential.

    Parameters
    ----------
    res : dict
        Output of `open_era5_day`, containing:
          - res['datasets']['2t']: xarray.Dataset with 2m temperature (K)
          - res['datasets']['q'] : xarray.Dataset with specific humidity (kg/kg)
          - res['zs']            : xarray.DataArray with surface geopotential height (m)

    Returns
    -------
    mse_da : xarray.DataArray
        Surface moist static energy (J/kg) with the same coordinates as the 2m temperature field.
    """

    # --- Extract variables ---
    ds_2t = res["datasets"]["tasmax"][["tasmax","lat", "lon", "time"]]
    ds_q  = res["datasets"]["q"][["q","lat", "lon", "time"]]
    zs    = res["zs"]

    # Try to find the variable names automatically
    var_2t = list(ds_2t.data_vars)[0]
    var_q  = list(ds_q.data_vars)[0]

    T = ds_2t[var_2t] * units.kelvin
    q = ds_q[var_q] * units.dimensionless
    z = mpcalc.geopotential_to_height(zs * units('m^2/s^2'))

    # --- Compute MSE using metpy ---
    mse = mpcalc.moist_static_energy(z, T, q)

    mse_ds = mse.to_dataset(name="mse")
    mse_ds.attrs.update({
        "long_name": "Surface moist static energy", "units": "J kg-1"
    })

    return mse_ds

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units

def compute_sat_mse(res, hPa = 500):
    """
    Compute surface moist static energy (MSE) from 2m temperature, specific humidity, and surface geopotential.

    Parameters
    ----------
    res : dict
        Output of `open_era5_day`, containing:
          - res['datasets']['2t']: xarray.Dataset with 2m temperature (K)
          - res['datasets']['q'] : xarray.Dataset with specific humidity (kg/kg)
          - res['zs']            : xarray.DataArray with surface geopotential height (m)

    Returns
    -------
    mse_da : xarray.DataArray
        Surface moist static energy (J/kg) with the same coordinates as the 2m temperature field.
    """

    # --- Extract variables ---
    ds_t  = res["datasets"]["t"][["t","lat", "lon", "time"]]
    ds_z  = res["datasets"]["z"][["z", "lat", "lon", "time"]]

    # Try to find the variable names automatically
    var_t = list(ds_t.data_vars)[0]
    var_z = list(ds_z.data_vars)[0]

    T = ds_t[var_t] * units.kelvin
    z = mpcalc.geopotential_to_height(ds_z[var_z] * units('m^2/s^2'))

    p_sat = mpcalc.saturation_vapor_pressure(T)
    q_sat = 0.622*p_sat/(500* units.hPa)

    # --- Compute MSE using metpy ---
    mse = mpcalc.moist_static_energy(z, T, q_sat)

    mse_ds = mse.to_dataset(name="mse_sat")
    mse_ds.attrs.update({
        "long_name": f"Saturated moist static energy at {hPa} hPa", "units": "J kg-1"
    })
    
    return mse_ds

In [ ]:
# Example use:
res             = open_era5_day("2014-06")
mse_surface     = compute_surface_mse(res)
mse_sat         = compute_sat_mse(res)
mse_sat


In [ ]:
import xarray as xr
import pandas as pd
from tqdm import tqdm

# === Edit these paths if different ===
final_path = "/work/uc1275/u301827/02_MSE/paris/raw/"

# Variable mapping
era5_vars = {
    130: "t",       # Temperature, pl @ 500 hPa
    129: "z",       # Geopotential, pl @ 500 hPa
    133: "q",       # Specific humidity, ml @ 137
    167: "tasmax",  # 2m max temperature
    166: "2t",      # 2m temperature
    168: "2d",      # 2m dewpoint temperature
    39 : "swvl1",   # Soil water layer 1
    #232: "ie",      # Surface flux variable
    #59 : "cape",    # Convective available potential energy
    159: "blh",     # Boundary layer height
    #182: "e",       # Evaporation
    #507: "pev",     # Potential evaporation
    #146: "sshf",    # Surface sensible heat flux
    #147: "slhf",    # Surface latent heat flux
    #49 : "10fg",     # 10m wind gust
    134: "sp"
}

months = pd.date_range("1940-01-01", "2024-12-01", freq="MS")
month_strings = [m.strftime("%Y-%m") for m in months]

# -------------------------------
# STORAGE LISTS
# -------------------------------
t_list, z_list, q_list = [], [], []
tasmax_list, t2_list, td2_list = [], [], []
swvl1_list, ie_list = [], []

sp_list, cape_list, blh_list = [], [], []
e_list, pev_list = [], []
sshf_list, slhf_list = [], []
fg10_list = []

mse_surface_list, mse_sat_list = [], []

# -------------------------------
# MAIN LOOP
# -------------------------------
for date_str in tqdm(month_strings):

    try:
        res = open_era5_day(date_str)  # contains res["datasets"]
        mse_surface = compute_surface_mse(res)
        mse_sat = compute_sat_mse(res)

        ds = res["datasets"]

        if "t" in ds:        t_list.append(ds["t"])
        if "z" in ds:        z_list.append(ds["z"])
        if "q" in ds:        q_list.append(ds["q"])
        if "tasmax" in ds:   tasmax_list.append(ds["tasmax"])
        if "2t" in ds:       t2_list.append(ds["2t"])
        if "2d" in ds:       td2_list.append(ds["2d"])
        if "swvl1" in ds:    swvl1_list.append(ds["swvl1"])
        #if "ie" in ds:       ie_list.append(ds["ie"])

        #if "cape" in ds:     cape_list.append(ds["cape"])
        if "blh" in ds:      blh_list.append(ds["blh"])
        #if "e" in ds:        e_list.append(ds["e"])
        #if "pev" in ds:      pev_list.append(ds["pev"])
        #if "sshf" in ds:     sshf_list.append(ds["sshf"])
        #if "slhf" in ds:     slhf_list.append(ds["slhf"])
        #if "10fg" in ds:     fg10_list.append(ds["10fg"])
        if "sp" in ds:       sp_list.append(ds["sp"])

        mse_surface_list.append(mse_surface)
        mse_sat_list.append(mse_sat)

    except Exception as e:
        print(f"Skipping {date_str}: {e}")

# -------------------------------
# CONCAT ALL VARIABLES
# -------------------------------
def safe_concat(lst):
    return xr.concat(lst, dim="time") if len(lst) > 0 else None

t_all       = safe_concat(t_list)
z_all       = safe_concat(z_list)
q_all       = safe_concat(q_list)
tasmax_all  = safe_concat(tasmax_list)
t2_all      = safe_concat(t2_list)
td2_all     = safe_concat(td2_list)
swvl1_all   = safe_concat(swvl1_list)
ie_all      = safe_concat(ie_list)

cape_all    = safe_concat(cape_list)
blh_all     = safe_concat(blh_list)
e_all       = safe_concat(e_list)
pev_all     = safe_concat(pev_list)
sshf_all    = safe_concat(sshf_list)
slhf_all    = safe_concat(slhf_list)
fg10_all    = safe_concat(fg10_list)
sp_all      = safe_concat(sp_list)

mse_surface_all = safe_concat(mse_surface_list)
mse_sat_all     = safe_concat(mse_sat_list)

# -------------------------------
# Infer time coordinate
# -------------------------------
time_coord = t_all.time if t_all is not None else tasmax_all.time


In [ ]:
# -------------------------------
# BUILD FINAL DATASET
# -------------------------------
final_ds = xr.Dataset(
    {
        "t"        : t_all.t,
        "z"        : z_all.z,
        "q"        : q_all.q,
        "tasmax"   : tasmax_all.tasmax,
        "t2m"      : t2_all["2t"],
        "td2m"     : td2_all["2d"],
        "swvl1"    : swvl1_all.swvl1,
        #"ie"       : ie_all.ie,
        "sp"       : sp_all.sp,

        #"cape"     : cape_all.cape,
        "blh"      : blh_all.blh,
        #"e"        : e_all.e,
        #"pev"      : pev_all.pev,
        #"sshf"     : sshf_all.sshf,
        #"slhf"     : slhf_all.slhf,
        #"10fg"     : fg10_all["10fg"],

        "mse"      : mse_surface_all.mse,
        "mse_sat"  : mse_sat_all.mse_sat,
    },
    coords={"time": time_coord.time.values}
)

# -------------------------------
# SAVE OUTPUT
# -------------------------------
outfile = "/work/uc1275/u301827/02_MSE/paris/era5_mse_full_1940_2024_new.nc"
final_ds.to_netcdf(outfile)

print("Saved:", outfile)
